# PASO 1: Análisis Exploratorio de Datos (EDA)
## Predicción de Deserción Estudiantil
Dataset: UCI - Predict Students' Dropout and Academic Success

**Objetivo:** Explorar el dataset, entender distribuciones, correlaciones y detectar outliers.

In [ ]:
# Instalación de librerías necesarias
!pip install -q pandas numpy matplotlib seaborn scikit-learn scipy -U
!pip install -q xgboost shap groq python-dotenv

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configurar estilos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Librerías importadas correctamente")

### 1.1 Descargar el Dataset

In [ ]:
# Descargar el dataset directamente desde UCI
url = "https://archive.ics.uci.edu/static/public/697/data.csv"

try:
    df = pd.read_csv(url)
    print(f"✅ Dataset descargado exitosamente")
    print(f"📊 Tamaño del dataset: {df.shape}")
except Exception as e:
    print(f"⚠️ Error descargando desde UCI. Intenta usar este enlace: {url}")
    print(f"Error: {e}")

### 1.2 Resumen Estadístico Completo

In [ ]:
# Información general del dataset
print("="*80)
print("RESUMEN GENERAL DEL DATASET")
print("="*80)
print(f"\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\n🔍 Tipos de datos:")
print(df.dtypes)

print(f"\n⚠️ Valores nulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ No hay valores nulos")

print(f"\n📊 Estadísticas descriptivas:")
print(df.describe())

In [ ]:
# Información detallada de cada columna
print("\n" + "="*80)
print("INFORMACIÓN DETALLADA DE COLUMNAS")
print("="*80)
for col in df.columns:
    print(f"\n🔹 {col}")
    print(f"   Tipo: {df[col].dtype}")
    print(f"   Valores únicos: {df[col].nunique()}")
    if df[col].nunique() <= 10:
        print(f"   Valores: {df[col].unique()[:10]}")

### 1.3 Análisis de la Variable Objetivo

In [ ]:
# Identificar la columna Target
target_col = 'Target' if 'Target' in df.columns else df.columns[-1]
print(f"✅ Columna Target identificada: {target_col}")
print(f"\n📊 Distribución de clases:")
print(df[target_col].value_counts())
print(f"\n📈 Proporción (%)")
print(df[target_col].value_counts(normalize=True) * 100)

In [ ]:
# Gráfico de distribución del Target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
target_counts = df[target_col].value_counts()
colors = ['#e74c3c', '#2ecc71', '#3498db']
axes[0].bar(target_counts.index, target_counts.values, color=colors[:len(target_counts)])
axes[0].set_title(f'Distribución de {target_col}', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Frecuencia')
axes[0].grid(alpha=0.3)
# Añadir valores en barras
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Gráfico de pastel
axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors[:len(target_counts)], startangle=90)
axes[1].set_title(f'Proporción de {target_col}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/01_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 01_target_distribution.png")

### 1.4 Matriz de Correlación y Top Features

In [ ]:
# Preparar datos numéricos para correlación
df_numeric = df.select_dtypes(include=[np.number]).copy()

# Convertir Target a numérico si es categórico
if target_col in df.columns and df[target_col].dtype == 'object':
    target_mapping = {val: i for i, val in enumerate(df[target_col].unique())}
    df_numeric[target_col] = df[target_col].map(target_mapping)
    print(f"Mapeo de Target: {target_mapping}")

# Calcular correlación
correlacion = df_numeric.corr()
print(f"\n✅ Matriz de correlación calculada ({correlacion.shape})")

In [ ]:
# Top 15 features más correlacionadas con Target
if target_col in df_numeric.columns:
    corr_con_target = correlacion[target_col].abs().sort_values(ascending=False)
    print(f"\n📊 Top 15 features correlacionadas con {target_col}:")
    print(corr_con_target.head(16))  # 16 porque incluye el target mismo
    
    top_features = corr_con_target[1:16].index.tolist()  # Excluir el target
else:
    # Si no está en numéricos, seleccionar los más correlacionados
    print(f"⚠️ {target_col} no está en variables numéricas. Selectores basados en otras columnas.")

In [ ]:
# Heatmap de la matriz de correlación (todas las variables)
plt.figure(figsize=(16, 12))
sns.heatmap(correlacion, cmap='coolwarm', center=0, annot=False, 
            fmt='.2f', square=True, cbar_kws={'label': 'Correlación'})
plt.title('Matriz de Correlación - Todas las Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../figures/02_correlation_matrix_full.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 02_correlation_matrix_full.png")

In [ ]:
# Heatmap de Top 15 features
if len(top_features) > 0:
    corr_top = df_numeric[top_features + [target_col]].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_top, cmap='coolwarm', center=0, annot=True, 
                fmt='.2f', square=True, cbar_kws={'label': 'Correlación'})
    plt.title('Top 15 Features Correlacionadas con Target', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig('../figures/03_correlation_top15.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✅ Figura guardada: 03_correlation_top15.png")

### 1.5 Distribución de Top 6 Features Más Correlacionadas

In [ ]:
# Seleccionar top 6 features
top_6_features = top_features[:6] if len(top_features) >= 6 else top_features
print(f"✅ Top 6 Features seleccionadas: {top_6_features}")

# Crear gráficos de distribución
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, feature in enumerate(top_6_features):
    axes[idx].hist(df_numeric[feature], bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'Distribución de {feature}\n(Corr: {corr_con_target[feature]:.3f})', fontweight='bold')
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/04_top6_features_distribution.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 04_top6_features_distribution.png")

### 1.6 Detección de Outliers (Boxplots)

In [ ]:
# Calcular estadísticas de outliers
def detectar_outliers(data):
    """Detecta outliers usando IQR"""
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    outliers = data[(data < Q1 - 1.5*IQR) | (data > Q3 + 1.5*IQR)]
    return len(outliers), len(outliers) / len(data) * 100

print("\n📊 Análisis de Outliers (Método IQR):")
print("="*60)
for col in df_numeric.columns:
    if col == target_col:
        continue
    count, pct = detectar_outliers(df_numeric[col])
    if count > 0:
        print(f"{col}: {count} outliers ({pct:.2f}%)")

In [ ]:
# Boxplots de top 8 features
top_8_features = top_features[:8] if len(top_features) >= 8 else top_features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(top_8_features):
    axes[idx].boxplot(df_numeric[feature], vert=True)
    axes[idx].set_title(f'Boxplot: {feature}', fontweight='bold')
    axes[idx].set_ylabel('Valor')
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/05_boxplots_outliers.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 05_boxplots_outliers.png")

### 1.7 Análisis de Distribuciones por Clase (Target)

In [ ]:
# Distribuciones del Top 6 separadas por clase
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, feature in enumerate(top_6_features):
    for class_val in df[target_col].unique():
        mask = df[target_col] == class_val
        axes[idx].hist(df_numeric.loc[mask, feature], bins=25, alpha=0.6, label=class_val, edgecolor='black')
    
    axes[idx].set_title(f'{feature} por Clase de {target_col}', fontweight='bold')
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/06_features_by_class.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Figura guardada: 06_features_by_class.png")

### 1.8 Conclusiones del EDA

In [ ]:
# Generar reporte de conclusiones
conclusiones = f"""
═══════════════════════════════════════════════════════════════════════════════
CONCLUSIONES DEL ANÁLISIS EXPLORATORIO (EDA)
═══════════════════════════════════════════════════════════════════════════════

📊 CARACTERÍSTICAS GENERALES DEL DATASET:
  • Tamaño: {df.shape[0]:,} muestras × {df.shape[1]} características
  • Valores nulos: {df.isnull().sum().sum()} (sin faltantes críticos)
  • Variables numéricas: {len(df_numeric.columns)}
  • Rango temporal: Estudiantes de múltiples cohortes y programas

🎯 VARIABLE OBJETIVO (Target):
"""

# Agregar detalles del target
for clase in df[target_col].unique():
    count = (df[target_col] == clase).sum()
    pct = count / len(df) * 100
    conclusiones += f"  • {clase}: {count:,} casos ({pct:.1f}%)\n"

conclusiones += f"""
  ⚠️ DESBALANCE DE CLASES: {'Sí (requiere atención en modelado)' if max(df[target_col].value_counts()) / len(df) > 0.6 else 'Moderado'}

🔗 FEATURES MÁS RELEVANTES (Top 6 por Correlación):
"""

for i, feature in enumerate(top_6_features, 1):
    corr_val = corr_con_target[feature]
    conclusiones += f"  {i}. {feature}: {corr_val:.3f}\n"

conclusiones += f"""
🚨 OUTLIERS DETECTADOS:
  • Método IQR (1.5 × IQR)
  • Features con outliers: {sum(1 for col in df_numeric.columns if col != target_col and detectar_outliers(df_numeric[col])[0] > 0)}
  • Acción: Mantener outliers (pueden ser casos reales de deserción) o usar escalado robusto

📈 DISTRIBUCIONES:
  • Distribuciones varían significativamente entre clases
  • Sugiere que los features tienen poder predictivo
  • Preparar datos: escalado con StandardScaler es recomendado

✅ RECOMENDACIONES PARA MODELADO:
  1. Usar StandardScaler para normalizar features
  2. Considerar técnicas de manejo de desbalance (si aplica)
  3. Validar con XGBoost que es robusto a outliers
  4. Usar top features para explicabilidad con SHAP
  5. División: 70% train, 15% val, 15% test (random_state=42)

═══════════════════════════════════════════════════════════════════════════════
"""

print(conclusiones)

# Guardar conclusiones en archivo
with open('../reports/01_EDA_conclusions.txt', 'w', encoding='utf-8') as f:
    f.write(conclusiones)

print("\n✅ Conclusiones guardadas en: 01_EDA_conclusions.txt")

In [ ]:
# Guardar el dataset procesado para el siguiente paso
df.to_csv('../data/processed/dataset_completo.csv', index=False)
print(f"✅ Dataset guardado: {df.shape}")
print(f"✅ EDA completado. Se han generado 5 figuras PNG en la carpeta 'figures/'")